# 00 - Setup e Diagnostico do Ambiente

Objetivo: validar que o ambiente (Python, dependencias, ffmpeg, GPU, caminhos do dataset) esta pronto antes de iniciar o pipeline. Ver `PLANO_EXECUCAO_MINIQUALIFICACAO.md` (Secoes 1 e 7) para o contexto completo.

Criterio de saida: todas as celulas abaixo devem rodar sem excecao. Falhas aqui devem ser resolvidas antes de seguir para o Notebook 01.

In [ ]:
import sys
import platform

print("Python:", sys.version)
print("Plataforma:", platform.platform())

assert sys.version_info[:2] in [(3, 10), (3, 11)], (
    "Recomenda-se Python 3.10/3.11 em venv dedicado (ver Secao 7 do plano); "
    f"versao atual: {sys.version_info[:2]}"
)

In [ ]:
# Verifica dependencias criticas
import importlib

pkgs = ["cv2", "mediapipe", "pandas", "numpy", "sklearn", "xgboost", "imblearn", "torch", "matplotlib", "seaborn"]
faltando = []
for p in pkgs:
    try:
        m = importlib.import_module(p)
        print(f"OK  {p:12s} versao={getattr(m, '__version__', '?')}")
    except ImportError:
        faltando.append(p)
        print(f"FALTA {p}")

if faltando:
    print("\nInstale com: pip install -r requirements.txt")

In [ ]:
# Verifica GPU (opcional) - pipeline deve funcionar tambem sem GPU (ver Secao 1 do plano)
try:
    import torch
    print("CUDA disponivel:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("Sem GPU -> pipeline vai rodar em CPU (ver estrategia leve na Secao 5 do plano)")
except ImportError:
    print("torch nao instalado ainda")

In [ ]:
# Verifica ffmpeg no PATH
import shutil

ffmpeg_path = shutil.which("ffmpeg")
print("ffmpeg encontrado em:", ffmpeg_path if ffmpeg_path else "NAO ENCONTRADO - instale via choco/winget")

In [ ]:
# Valida estrutura do dataset e caminhos
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
DATASET_DIR = ROOT / "datasets" / "DAiSEE"
LABELS_DIR = DATASET_DIR / "Labels"
VIDEOS_DIR = DATASET_DIR / "DataSet"

for p in [DATASET_DIR, LABELS_DIR, VIDEOS_DIR]:
    print(p, "->", "OK" if p.exists() else "NAO ENCONTRADO")

for split in ["TrainLabels.csv", "ValidationLabels.csv", "TestLabels.csv", "AllLabels.csv"]:
    f = LABELS_DIR / split
    print(f, "->", "OK" if f.exists() else "NAO ENCONTRADO")

In [ ]:
# Sanity check: localizar um video de exemplo a partir dos rotulos
import pandas as pd

df = pd.read_csv(LABELS_DIR / "TrainLabels.csv")
df.columns = [c.strip() for c in df.columns]
print(df.head())

sample_clip = df.iloc[0]["ClipID"]
user_id = sample_clip[:6]
clip_stub = sample_clip.replace(".avi", "")
sample_path = VIDEOS_DIR / "Train" / user_id / clip_stub / sample_clip
print("Video de exemplo:", sample_path, "->", "OK" if sample_path.exists() else "NAO ENCONTRADO")

## Checklist de saida
- [ ] Python 3.10/3.11 em uso
- [ ] Todas as dependencias do `requirements.txt` instaladas
- [ ] ffmpeg encontrado
- [ ] Pastas do dataset e arquivos de rotulos encontrados
- [ ] Pelo menos um video de exemplo localizado com sucesso

Se tudo OK, seguir para `01_analise_exploratoria.ipynb`.